<a href="https://colab.research.google.com/github/zachedwards2/econ8310-assignment1/blob/main/assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

train_url = "https://github.com/dustywhite7/econ8310-assignment1/raw/main/assignment_data_train.csv"
data = pd.read_csv(train_url, parse_dates=['Timestamp'])
data.set_index('Timestamp', inplace=True)
data.index.freq = 'h'

# Weekly seasonality: 24*7=168 hours
model = ExponentialSmoothing(
    data['trips'],
    trend='add',
    damped_trend=True,
    seasonal='add',
    seasonal_periods=168
)


modelFit = model.fit(
    smoothing_level=0.2,
    smoothing_slope=0.1,
    smoothing_seasonal=0.2,
    damping_slope=0.9,
    optimized=False
)


pred = modelFit.forecast(steps=744)
pred[pred < 0] = 0

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=data.index, y=data['trips'],
    mode='lines', name='Actual', line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=data.index, y=modelFit.fittedvalues,
    mode='lines', name='Fitted', line=dict(color='green')
))

fig.add_trace(go.Scatter(
    x=pred.index, y=pred,
    mode='lines', name='Forecast', line=dict(color='red')
))

fig.update_layout(
    title='NYC Taxi Trips Forecast',
    xaxis_title='Timestamp',
    yaxis_title='Number of Trips',
    legend=dict(x=0.01, y=0.99)
)

fig.show()




/tmp/ipython-input-3786992941.py:19: FutureWarning:

the 'smoothing_slope' keyword is deprecated, use 'smoothing_trend' instead.

/usr/local/lib/python3.12/dist-packages/pandas/util/_decorators.py:213: FutureWarning:

the 'damping_slope' keyword is deprecated, use 'damping_trend' instead.



In [52]:
import pandas as pd
import plotly.express as px
from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing

data = pd.read_csv("https://github.com/dustywhite7/Econ8310/raw/master/DataSets/RecessionForecasting.csv")
data['DATE'] = pd.to_datetime(data['DATE'])

px.line(data, x="DATE", y='CivEmpLevel')

employment = data['CivEmpLevel']
employment.index = data['DATE']
employment.index.freq = employment.index.inferred_freq

alpha020 = SimpleExpSmoothing(employment).fit(
                                        smoothing_level=0.2,
                                        optimized=False)

alpha050 = SimpleExpSmoothing(employment).fit(
                                        smoothing_level=0.5,
                                        optimized=False)

alpha080 = SimpleExpSmoothing(employment).fit(
                                        smoothing_level=0.8,
                                        optimized=False)

forecast020 = alpha020.forecast(3)
forecast050 = alpha050.forecast(3)
forecast080 = alpha080.forecast(3)

import plotly.graph_objects as go

# Plotting our data

smoothData = pd.DataFrame([employment.values, alpha020.fittedvalues.values,  alpha050.fittedvalues.values,  alpha080.fittedvalues.values]).T
smoothData.columns = ['Truth', 'alpha=0.2', 'alpha=0.5', 'alpha=0.8']
smoothData.index = employment.index

fig = px.line(smoothData, y = ['Truth', 'alpha=0.2', 'alpha=0.5', 'alpha=0.8'],
        x = smoothData.index,
        color_discrete_map={"Truth": 'blue',
                           'alpha=0.2': 'red',
                            'alpha=0.5':'green',
                            'alpha=0.8':'purple'}
       )

fig.update_xaxes(range=[smoothData.index[-50], forecast020.index[-1]])
fig.update_yaxes(range=[142000, 153000])


# Incorporating the Forecasts

fig.add_trace(go.Scatter(x=forecast020.index, y = forecast020.values, name='Forecast alpha=0.2', line={'color':'red'}))
fig.add_trace(go.Scatter(x=forecast050.index, y = forecast050.values, name='Forecast alpha=0.5', line={'color':'green'}))
fig.add_trace(go.Scatter(x=forecast080.index, y = forecast080.values, name='Forecast alpha=0.8', line={'color':'purple'}))
